### Create Population-Weighted Name Frequency Tables

Builds two tables estimating "how many living humans have this name":
- `pop_freq_last` — last/family names
- `pop_freq_given` — given/first names (also used for middle names)

**Sources** (in priority order):
1. **Chinese surnames** — `name_freq_chinese_surnames` (1,806 chars, 1.18B people). Romanized via unidecode. Mandarin/Cantonese split ~93%/7% for top surnames.
2. **Japanese surnames** — `name_freq_japanese_surnames` (5,000 kanji, 112M). Romanized via unidecode.
3. **Korean surnames** — `name_freq_korean_surnames` (100, already romanized, 49M).
4. **census.names** — `name_freq_census_lastnames`/`firstnames` (~120 countries). Excludes CN/JP/KR.
5. **Chinese given names** — `name_freq_chinese_pinyin` (93K, already in pinyin).
6. **Japanese given names** — `name_freq_japanese_given` (15K, already romanized).

Each source is weighted by its country/region's share of world population.

Job: #105.12 name-freq-population-estimate

In [ ]:
%pip install unidecode

In [ ]:
import re
from collections import defaultdict
from unidecode import unidecode

import pandas as pd
from pyspark.sql import functions as F

dbutils.widgets.text("env_suffix", "", "Environment Suffix")
env_suffix = dbutils.widgets.get("env_suffix")

WORLD_POP = 8_100_000_000

# Cantonese romanization mapping for top Chinese surnames.
# These surnames have well-known Cantonese forms used by HK/Guangdong diaspora.
# Population split: ~93% Mandarin speakers, ~7% Cantonese speakers.
CANTONESE_MAP = {
    '王': 'wong', '张': 'cheung', '刘': 'lau', '陈': 'chan',
    '杨': 'yeung', '赵': 'chiu', '吴': 'ng', '周': 'chow',
    '郭': 'kwok', '何': 'ho', '林': 'lam', '梁': 'leung',
    '谢': 'tse', '冯': 'fung', '曾': 'tsang', '钟': 'chung',
    '谭': 'tam',
}
MANDARIN_PCT = 0.93
CANTONESE_PCT = 0.07


def romanize(text):
    """Romanize a string to lowercase ASCII, keeping only letters."""
    return re.sub(r'[^a-z]', '', unidecode(text).lower().strip())

#### Step 1: Build Chinese surname frequencies (romanized, Mandarin/Cantonese split)

In [ ]:
chinese_surnames_df = spark.table(f"openalex{env_suffix}.authors.name_freq_chinese_surnames").toPandas()
chinese_census_pop = chinese_surnames_df['n_total'].sum()
print(f"Chinese surname source: {len(chinese_surnames_df):,} characters, {chinese_census_pop:,} people")

# Romanize and split Mandarin/Cantonese
chinese_last = defaultdict(float)
for _, row in chinese_surnames_df.iterrows():
    char = row['surname']
    # Use proportion, not absolute count, so we can apply to current population
    proportion = row['n_total'] / chinese_census_pop
    
    mandarin = romanize(char)
    if char in CANTONESE_MAP:
        cantonese = CANTONESE_MAP[char]
        chinese_last[mandarin] += proportion * MANDARIN_PCT
        chinese_last[cantonese] += proportion * CANTONESE_PCT
    else:
        chinese_last[mandarin] += proportion

print(f"Unique romanized surnames: {len(chinese_last):,}")
print(f"Sum of proportions: {sum(chinese_last.values()):.4f} (should be ~1.0)")
print(f"\nTop 10:")
for name, prop in sorted(chinese_last.items(), key=lambda x: -x[1])[:10]:
    print(f"  {name}: {prop:.4f} ({prop * 1_425_900_000:,.0f} est. people)")

#### Step 2: Build Japanese surname frequencies (romanized)

In [ ]:
japanese_surnames_df = spark.table(f"openalex{env_suffix}.authors.name_freq_japanese_surnames").toPandas()
japanese_surname_pop = japanese_surnames_df['population'].sum()
print(f"Japanese surname source: {len(japanese_surnames_df):,} entries, {japanese_surname_pop:,} people")

# Romanize via unidecode and aggregate
japanese_last = defaultdict(float)
for _, row in japanese_surnames_df.iterrows():
    romaji = romanize(row['surname_kanji'])
    if romaji:  # skip empty
        japanese_last[romaji] += row['population'] / japanese_surname_pop

print(f"Unique romanized surnames: {len(japanese_last):,}")
print(f"Sum of proportions: {sum(japanese_last.values()):.4f}")
print(f"\nTop 10:")
for name, prop in sorted(japanese_last.items(), key=lambda x: -x[1])[:10]:
    print(f"  {name}: {prop:.4f} ({prop * 123_300_000:,.0f} est. people)")

#### Step 3: Build Korean surname frequencies (already romanized)

In [ ]:
korean_surnames_df = spark.table(f"openalex{env_suffix}.authors.name_freq_korean_surnames").toPandas()
korean_surname_pop = korean_surnames_df['population_2015'].sum()
print(f"Korean surname source: {len(korean_surnames_df):,} entries, {korean_surname_pop:,} people")

korean_last = {}
for _, row in korean_surnames_df.iterrows():
    name = row['romanized'].lower().strip()
    korean_last[name] = row['population_2015'] / korean_surname_pop

print(f"Sum of proportions: {sum(korean_last.values()):.4f}")
print(f"\nTop 10:")
for name, prop in sorted(korean_last.items(), key=lambda x: -x[1])[:10]:
    print(f"  {name}: {prop:.4f} ({prop * 51_700_000:,.0f} est. people)")

#### Step 4: Census.names last name frequencies (per-country, population-weighted)

For each country in census.names (excluding CN, JP, KR):
- `P(name | country) = name_freq / country_sample_total`
- Contribution to global: `P(name | country) × (country_pop / world_pop)`

In [ ]:
# Get country populations
country_pop_df = spark.table(f"openalex{env_suffix}.authors.country_population").toPandas()
country_pop = dict(zip(country_pop_df['country_code'], country_pop_df['population_2024']))

# Get census.names last name data, excluding CN, JP, KR (we have better sources)
census_last_df = spark.sql(f"""
    SELECT name_ascii, country_code, frequency
    FROM openalex{env_suffix}.authors.name_freq_census_lastnames
    WHERE country_code NOT IN ('CN', 'JP', 'KR')
""").toPandas()

# Compute per-country sample totals
country_sample_totals = census_last_df.groupby('country_code')['frequency'].sum().to_dict()

print(f"Census.names countries (excl CN/JP/KR): {len(country_sample_totals)}")
print(f"Total name-country rows: {len(census_last_df):,}")

# Population-weighted frequencies
census_last = defaultdict(float)
for _, row in census_last_df.iterrows():
    cc = row['country_code']
    name = row['name_ascii'].lower().strip()
    freq = row['frequency']
    sample_total = country_sample_totals[cc]
    pop = country_pop.get(cc, 0)
    
    if sample_total > 0 and pop > 0:
        p_name_given_country = freq / sample_total
        country_weight = pop / WORLD_POP
        census_last[name] += p_name_given_country * country_weight

census_pop_covered = sum(country_pop.get(cc, 0) for cc in country_sample_totals)
print(f"Population covered by census.names (excl CJK): {census_pop_covered:,}")
print(f"Unique surnames from census.names: {len(census_last):,}")
print(f"\nTop 10 from census.names:")
for name, prob in sorted(census_last.items(), key=lambda x: -x[1])[:10]:
    print(f"  {name}: {prob:.6f} ({prob * WORLD_POP:,.0f} est. people)")

#### Step 5: Combine all sources into `pop_freq_last`

For each name: `P(name) = P(name|China)×P(China) + P(name|Japan)×P(Japan) + P(name|Korea)×P(Korea) + Σ_census_country[...]`

In [ ]:
# Country/region weights (share of world population)
CHINA_POP = 1_425_900_000
JAPAN_POP = 123_300_000
KOREA_POP = 51_700_000

china_weight = CHINA_POP / WORLD_POP
japan_weight = JAPAN_POP / WORLD_POP
korea_weight = KOREA_POP / WORLD_POP

print(f"Weights: China={china_weight:.4f}, Japan={japan_weight:.4f}, Korea={korea_weight:.4f}")

# Combine all sources
combined_last = defaultdict(float)

# Chinese contribution
for name, prop in chinese_last.items():
    combined_last[name] += prop * china_weight

# Japanese contribution
for name, prop in japanese_last.items():
    combined_last[name] += prop * japan_weight

# Korean contribution
for name, prop in korean_last.items():
    combined_last[name] += prop * korea_weight

# Census.names contribution (already population-weighted)
for name, prob in census_last.items():
    combined_last[name] += prob

total_prob = sum(combined_last.values())
print(f"\nTotal names: {len(combined_last):,}")
print(f"Sum of all probabilities: {total_prob:.4f}")
print(f"  (should be < 1.0 — remainder is uncovered population + long-tail names)")

# Sanity checks: how many living humans have each name?
print(f"\n--- Sanity checks ---")
checks = ['wang', 'wong', 'li', 'smith', 'kim', 'lee', 'tanaka', 'garcia', 
           'nguyen', 'kumar', 'priem', 'johnson', 'muller', 'sato']
for name in checks:
    prob = combined_last.get(name, 0)
    n_humans = prob * WORLD_POP
    print(f"  {name}: {n_humans:>14,.0f} people ({prob:.6f})")

In [ ]:
# Write to Databricks table
last_name_table = f"openalex{env_suffix}.authors.pop_freq_last"

rows = [(name, prob, round(prob * WORLD_POP)) 
        for name, prob in combined_last.items()]
last_df = spark.createDataFrame(rows, ['name', 'prob', 'est_humans'])

last_df.write.format('delta').mode('overwrite').option(
    'overwriteSchema', 'true'
).saveAsTable(last_name_table)

written = spark.table(last_name_table).count()
print(f"Wrote {written:,} rows to {last_name_table}")

# Top 20
spark.sql(f"SELECT name, prob, est_humans FROM {last_name_table} ORDER BY prob DESC LIMIT 20").show(truncate=False)

---
### Given Name Frequency Table

Same approach: Chinese pinyin + Japanese given + census.names, population-weighted.

#### Step 6: Chinese given name frequencies

In [ ]:
chinese_given_df = spark.table(f"openalex{env_suffix}.authors.name_freq_chinese_pinyin").toPandas()
chinese_given_pop = chinese_given_df['n_total'].sum()
print(f"Chinese given names: {len(chinese_given_df):,} entries, {chinese_given_pop:,} people")

chinese_given = {}
for _, row in chinese_given_df.iterrows():
    name = row['pinyin_name'].lower().strip()
    if name:
        chinese_given[name] = row['n_total'] / chinese_given_pop

print(f"Unique given names: {len(chinese_given):,}")
print(f"Sum of proportions: {sum(chinese_given.values()):.4f}")
print(f"\nTop 10:")
for name, prop in sorted(chinese_given.items(), key=lambda x: -x[1])[:10]:
    print(f"  {name}: {prop:.4f}")

#### Step 7: Japanese given name frequencies

In [ ]:
# Check what the japanese_given table looks like
japanese_given_raw = spark.table(f"openalex{env_suffix}.authors.name_freq_japanese_given").toPandas()
print(f"Schema: {list(japanese_given_raw.columns)}")
print(f"Rows: {len(japanese_given_raw):,}")
print(japanese_given_raw.head(10))

In [ ]:
# The japanese_given table is already in the unified first table with romanized names.
# Let's use it from the unified table directly.
japanese_given_unified = spark.sql(f"""
    SELECT name, freq, total_pop, prob 
    FROM openalex{env_suffix}.authors.name_freq_first_unified 
    WHERE source = 'japanese_given'
""").toPandas()

japanese_given_total = japanese_given_unified['total_pop'].iloc[0] if len(japanese_given_unified) > 0 else 125_000_000
print(f"Japanese given names (from unified): {len(japanese_given_unified):,} entries, total_pop={japanese_given_total:,}")

japanese_given = {}
for _, row in japanese_given_unified.iterrows():
    name = row['name'].lower().strip()
    if name:
        japanese_given[name] = row['prob']  # already freq/total_pop

print(f"Sum of proportions: {sum(japanese_given.values()):.4f}")
print(f"\nTop 10:")
for name, prop in sorted(japanese_given.items(), key=lambda x: -x[1])[:10]:
    print(f"  {name}: {prop:.4f}")

#### Step 8: Census.names given name frequencies (per-country, population-weighted)

In [ ]:
# Exclude CN and JP (we have better sources). KR doesn't have a dedicated given name source,
# so we keep KR in census.names for given names.
census_given_df = spark.sql(f"""
    SELECT name_ascii, country_code, frequency
    FROM openalex{env_suffix}.authors.name_freq_census_firstnames
    WHERE country_code NOT IN ('CN', 'JP')
""").toPandas()

country_given_totals = census_given_df.groupby('country_code')['frequency'].sum().to_dict()
print(f"Census.names countries for given names (excl CN/JP): {len(country_given_totals)}")

census_given = defaultdict(float)
for _, row in census_given_df.iterrows():
    cc = row['country_code']
    name = row['name_ascii'].lower().strip()
    freq = row['frequency']
    sample_total = country_given_totals[cc]
    pop = country_pop.get(cc, 0)
    
    if sample_total > 0 and pop > 0:
        p_name_given_country = freq / sample_total
        country_weight = pop / WORLD_POP
        census_given[name] += p_name_given_country * country_weight

print(f"Unique given names from census.names: {len(census_given):,}")
print(f"\nTop 10:")
for name, prob in sorted(census_given.items(), key=lambda x: -x[1])[:10]:
    print(f"  {name}: {prob:.6f} ({prob * WORLD_POP:,.0f} est. people)")

#### Step 9: Combine into `pop_freq_given`

In [ ]:
combined_given = defaultdict(float)

# Chinese given names
for name, prop in chinese_given.items():
    combined_given[name] += prop * china_weight

# Japanese given names
for name, prop in japanese_given.items():
    combined_given[name] += prop * japan_weight

# Census.names (already population-weighted)
for name, prob in census_given.items():
    combined_given[name] += prob

total_given_prob = sum(combined_given.values())
print(f"Total given names: {len(combined_given):,}")
print(f"Sum of all probabilities: {total_given_prob:.4f}")

# Sanity checks
print(f"\n--- Sanity checks ---")
checks = ['muhammad', 'maria', 'wei', 'john', 'david', 'yuki', 'jason',
           'ahmed', 'jose', 'michael', 'hiroshi', 'kumar']
for name in checks:
    prob = combined_given.get(name, 0)
    n_humans = prob * WORLD_POP
    print(f"  {name}: {n_humans:>14,.0f} people ({prob:.6f})")

In [ ]:
# Write to Databricks table
given_name_table = f"openalex{env_suffix}.authors.pop_freq_given"

rows = [(name, prob, round(prob * WORLD_POP))
        for name, prob in combined_given.items()]
given_df = spark.createDataFrame(rows, ['name', 'prob', 'est_humans'])

given_df.write.format('delta').mode('overwrite').option(
    'overwriteSchema', 'true'
).saveAsTable(given_name_table)

written = spark.table(given_name_table).count()
print(f"Wrote {written:,} rows to {given_name_table}")

# Top 20
spark.sql(f"SELECT name, prob, est_humans FROM {given_name_table} ORDER BY prob DESC LIMIT 20").show(truncate=False)

---
### Verification

In [ ]:
print("=== Last Name Table ===")
last_stats = spark.sql(f"""
    SELECT 
        COUNT(*) as total_names,
        SUM(prob) as sum_prob,
        SUM(est_humans) as sum_humans,
        MAX(prob) as max_prob,
        MIN(prob) as min_prob
    FROM {last_name_table}
""").show(truncate=False)

print("\n=== Given Name Table ===")
given_stats = spark.sql(f"""
    SELECT 
        COUNT(*) as total_names,
        SUM(prob) as sum_prob,
        SUM(est_humans) as sum_humans,
        MAX(prob) as max_prob,
        MIN(prob) as min_prob
    FROM {given_name_table}
""").show(truncate=False)

In [ ]:
# Cross-check: do the most common names worldwide make sense?
print("Top 30 last names worldwide (by est. humans):")
spark.sql(f"""
    SELECT name, est_humans, ROUND(prob * 100, 3) as pct_of_world
    FROM {last_name_table} ORDER BY prob DESC LIMIT 30
""").show(truncate=False)

print("\nTop 30 given names worldwide (by est. humans):")
spark.sql(f"""
    SELECT name, est_humans, ROUND(prob * 100, 3) as pct_of_world
    FROM {given_name_table} ORDER BY prob DESC LIMIT 30
""").show(truncate=False)